# mAP scatters (complex + gene KO) + GSEA enrichment

Two scatter panels and a GSEA bar plot comparing the cell-dino phase-only OPS embedding against the sVAEplus crop-seq embedding.

1. **Complex mAP scatter** — per-complex `mean_average_precision` from crop-seq vs cell-dino; top-K most-divergent complexes labelled by complex name.
2. **Gene-KO mAP scatter** — per-perturbation mAP, crop-seq vs cell-dino; top-K labelled by gene symbol.
3. **GSEA bar plot** — top imaging- and RNA-favoured GO slim BP / CC terms from a prerank GSEA on the per-gene mAP difference, FDR-shaded.

Inputs live under `../../data/figures/figure_5/`. Panels are saved as `.pdf` + `.svg` into `../../output/figure_5/`.

## GSEA design notes

**Ranking metric — raw mAP difference (`cell_dino − crop_seq`).** The question is whether genes better resolved by one modality differ biologically from those better resolved by the other, so the score has to keep its absolute meaning: negative means crop-seq genuinely resolves that gene better. The systematic +0.13 offset (cell-dino wins for 704/1000 genes) is treated as real signal about the two assays, not as a batch effect to normalise away. Because the score keeps its sign, GSEA's negative tail contains only genes where RNA actually wins — the cell asserts this, checking that no leading-edge gene of any negative-NES term has a positive difference.

A `'rank'` metric (difference of within-modality percentile ranks) is computed alongside as a **diagnostic only**. It is considerably more sensitive (min q 0.019 vs 0.067; 5 terms at q<0.05 vs 0) because it removes the offset, but it breaks the sign guarantee: a gene imaging resolves better in absolute terms can land on the RNA side if its crop-seq percentile happens to be higher. Useful as a cross-check — it agrees on mRNA metabolic process, snRNA metabolic process and chromatin organization — but not as the published statistic.

**Vocabulary — GO slim, not full GO.** Against this 1000-gene library the full Enrichr `GO_Biological_Process_2025` + `GO_Cellular_Component_2025` yields 289 testable sets, ~30% of which are near-duplicates of another set (single-linkage Jaccard ≥ 0.5), with a median of only 16 library genes each. That is too coarse a filter and too fine a granularity at once: the RNA side produced nine near-identical spliceosome rows, and the imaging side — whose signal is spread thinly across many genes — produced almost nothing. `goslim_generic` gives 67 testable sets, 3% redundant, median 32 genes, covering 940/1000 genes.

Alternatives measured and rejected: MSigDB C5 GO:BP (959 sets, 65% redundant — worse than Enrichr GO on both axes), MSigDB/Enrichr Reactome (68–76% redundant), MSigDB Hallmark (only 20 testable sets, covers 268/1000 genes), CORUM (7 testable sets), `goslim_agr` (35 sets — over-collapsed), and GO DAG depth filtering (depth 3–4 gives 66 sets but less power).

**Regenerating the slim GMTs.** `GO_slim_generic_{BP,CC}_2026-06-15.gmt` were built from GO release `2026-06-15` (`go-basic.obo`) plus `goa_human.gaf` (`2026-05-21`), mapping each annotation to its most specific slim ancestor with `goatools.mapslim.mapslim`, dropping `NOT`-qualified annotations, and keeping slim terms with ≥5 human genes. They are whole-proteome, not library-restricted. **These two files need adding to the next Zenodo release** — `data/` is gitignored, so a fresh clone + Zenodo download will not have them.

**Other choices.**
- FDR is pooled across BP + CC, so the two panels share one multiple-testing correction.
- The background universe is the 1000-gene CRISPR library, not the genome. Gene-set permutation makes that the right null.
- The colour scale is **fixed** (q=1 → q=0.001) with q=0.05 marked, and each bar is annotated with its q. Do not rescale it to the data's own minimum.
- Under the raw metric only 4 of 67 terms have a negative NES at all, so the RNA side of the bar plot is short by construction — 704/1000 genes favour imaging, so few sets can concentrate in the negative tail. That asymmetry is the result, not a plotting bug.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from adjustText import adjust_text


# ---- Paper-wide plotting parameters (inlined from former parameters.py) ----
class P:
    # Output / sizing
    DPI                  = 300
    SAVEFIG_FORMAT       = 'svg'
    PANEL_SIZE_IN        = (2.4, 2.0)
    SQUARE_PANEL_SIZE_IN = (2.4, 2.4)

    # Fonts (panels are small so default sizes are small too)
    FONT_FAMILY           = 'Arial'
    FONT_SIZE_AXIS_LABEL  = 8
    FONT_SIZE_TICK        = 6
    FONT_SIZE_TITLE       = 10
    FONT_SIZE_LEGEND      = 8
    FONT_SIZE_PANEL_LABEL = 14

    # Colours
    COLOR_BACKGROUND  = '#bdbdbd'   # grey, used for null / "all" distributions
    COLOR_HIGHLIGHT   = '#e07a3a'   # orange, RNA-side
    COLOR_HIGHLIGHT_2 = '#1f8a8a'   # teal, image-side
    COLOR_TEXT        = '#222222'
    COLOR_AXIS        = '#222222'

    # Lines
    LINEWIDTH      = 1.4
    KDE_LINEWIDTH  = 1.8
    LINESTYLE_RNA  = '--'
    LINESTYLE_IMG  = '-'

    @staticmethod
    def apply_style():
        plt.rcParams.update({
            'font.family':        P.FONT_FAMILY,
            'font.size':          P.FONT_SIZE_AXIS_LABEL,
            'axes.labelsize':     P.FONT_SIZE_AXIS_LABEL,
            'axes.titlesize':     P.FONT_SIZE_TITLE,
            'xtick.labelsize':    P.FONT_SIZE_TICK,
            'ytick.labelsize':    P.FONT_SIZE_TICK,
            'legend.fontsize':    P.FONT_SIZE_LEGEND,
            'axes.linewidth':     P.LINEWIDTH,
            'xtick.major.width':  P.LINEWIDTH,
            'ytick.major.width':  P.LINEWIDTH,
            'text.color':         P.COLOR_TEXT,
            'axes.edgecolor':     P.COLOR_AXIS,
            'axes.labelcolor':    P.COLOR_TEXT,
            'xtick.color':        P.COLOR_AXIS,
            'ytick.color':        P.COLOR_AXIS,
            'svg.fonttype':       'none',  # keep text editable in Illustrator
            'pdf.fonttype':       42,
            'ps.fonttype':        42,
        })

P.apply_style()


FIGURE_DATA = Path("../../data/figures/figure_5")
FIGURES_DIR = Path("../../output/figure_5")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CROPSEQ_MAP          = FIGURE_DATA / "cropseq_ebi_map.csv"
IMAGE_MAP            = FIGURE_DATA / "celldino_phase_only_ebi.csv"
IMAGE_MAP_PER_GENE   = FIGURE_DATA / "celldino_phase_only_distinctiveness.csv"
CROPSEQ_MAP_PER_GENE = FIGURE_DATA / "svaeplus_distinctiveness_std_ntc.csv"
COMPLEX_YAML         = FIGURE_DATA / "EBI_complexes_v1_updated_gene_names.yaml"

TOP_K = 8
POINT_COLOR = P.COLOR_HIGHLIGHT_2          # teal, neutral between the two modalities
DIAG_COLOR  = '#888888'
LABEL_FONT_SIZE = 5

_COMPLEX_RE = re.compile(r'\bcomplete\s+complex\b|\bcomplex\b', re.IGNORECASE)
def clean_complex_name(name):
    """Strip the redundant 'complex' suffix and tidy whitespace/punctuation."""
    return re.sub(r'\s+', ' ', _COMPLEX_RE.sub('', name)).strip(' ,-')


def save_panel(fig, stem):
    """Save fig as both .pdf and .svg into FIGURES_DIR."""
    for ext in ('pdf', 'svg'):
        out = FIGURES_DIR / f'{stem}.{ext}'
        fig.savefig(out, dpi=P.DPI, bbox_inches='tight')
        print(f'  wrote {out}')

In [ ]:
# ---- mAP table (crop-seq vs image / cell_dino) — joined on complex_num,
#      decorated with complex name + member count from the EBI YAML ----
with open(COMPLEX_YAML) as f:
    chad_yaml = yaml.safe_load(f)
num_to_name    = {int(k): v['name'] for k, v in chad_yaml.items()}
num_to_members = {int(k): len(v['genes']) for k, v in chad_yaml.items()}

cs_map = pd.read_csv(CROPSEQ_MAP)
im_map = pd.read_csv(IMAGE_MAP)

shared = set(cs_map['complex_num']) & set(im_map['complex_num'])
cs_sub = (cs_map[cs_map['complex_num'].isin(shared)]
            .set_index('complex_num')[['mean_average_precision']]
            .rename(columns={'mean_average_precision': 'crop_seq'}))
im_sub = (im_map[im_map['complex_num'].isin(shared)]
            .set_index('complex_num')[['mean_average_precision']]
            .rename(columns={'mean_average_precision': 'cell_dino'}))
map_df = cs_sub.join(im_sub)
map_df['name']      = map_df.index.map(num_to_name)
map_df['n_members'] = map_df.index.map(num_to_members)
print(f'shared complexes in mAP scatter: {len(map_df)}')

In [ ]:
# ---- Complex mAP scatter (points sized by member count) ----
def members_to_size(n_members, s_min=8, s_max=80):
    vmin, vmax = n_members.min(), n_members.max()
    if vmin == vmax:
        return pd.Series(s_min, index=n_members.index)
    return s_min + (n_members - vmin) / (vmax - vmin) * (s_max - s_min)

sizes = members_to_size(map_df['n_members'])

fig, ax = plt.subplots(figsize=P.SQUARE_PANEL_SIZE_IN, dpi=P.DPI)
ax.scatter(map_df['crop_seq'], map_df['cell_dino'], s=sizes,
           color=POINT_COLOR, alpha=0.65, linewidths=0)
ax.plot([0, 1], [0, 1], linestyle='--', color=DIAG_COLOR, linewidth=0.8, alpha=0.7)

map_df = map_df.assign(diff=(map_df['crop_seq'] - map_df['cell_dino']).abs())
top = map_df.nlargest(TOP_K, 'diff').reset_index()

# Extra headroom above/below the data so labels can stagger across 2 rows.
ax.set_xlim(-0.02, 1.05); ax.set_ylim(-0.02, 1.05)
ax.set_xticks([0, 0.5, 1.0]); ax.set_yticks([0, 0.5, 1.0])
ax.set_xlabel('crop-seq mAP')
ax.set_ylabel('cell-dino mAP')
ax.set_title(f'Protein Complex mAP (n={len(map_df)})', fontsize=P.FONT_SIZE_TITLE)
ax.set_aspect('equal')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Within each strip, sort labels by source x and stagger across 2 sub-rows so
# adjacent labels never share a y — their horizontal bounding boxes can bleed
# past each other without visually colliding.
def place_strip(rows, y_base, direction, row_step=0.09):
    if rows.empty:
        return
    rows = rows.sort_values('crop_seq').reset_index(drop=True)
    n = len(rows)
    label_xs = np.linspace(0.0, 1.05, n)
    for i, (x_lbl, (_, row)) in enumerate(zip(label_xs, rows.iterrows())):
        sub_row = i % 2                          # 0, 1, 0, 1, ...
        y_lbl = y_base + direction * sub_row * row_step
        # ax.annotate(
        #     clean_complex_name(row['name']),
        #     xy=(row['crop_seq'], row['cell_dino']),
        #     xytext=(x_lbl, y_lbl),
        #     fontsize=LABEL_FONT_SIZE, color='black',
        #     ha='center', va='center',
        #     arrowprops=dict(arrowstyle='-', color='black', lw=0.4,
        #                     shrinkA=0, shrinkB=2),
        # )

above = top[top['cell_dino'] >  top['crop_seq']]
below = top[top['cell_dino'] <= top['crop_seq']]
place_strip(above, y_base=1.12, direction=+1)
place_strip(below, y_base=-0.12, direction=-1)

fig.tight_layout()
save_panel(fig, 'mAP_complex_scatter')
plt.show()

In [ ]:
# ---- Gene-KO mAP scatter (crop-seq vs image) ----
cs_gene = (pd.read_csv(CROPSEQ_MAP_PER_GENE)[['perturbation', 'mean_average_precision']]
             .rename(columns={'mean_average_precision': 'crop_seq'}))
im_gene = (pd.read_csv(IMAGE_MAP_PER_GENE)[['perturbation', 'mean_average_precision']]
             .rename(columns={'mean_average_precision': 'cell_dino'}))
gene_df = (cs_gene.merge(im_gene, on='perturbation', how='inner')
                  .loc[lambda d: ~d['perturbation'].str.startswith('NTC')]
                  .reset_index(drop=True))
print(f'shared genes in per-gene mAP scatter: {len(gene_df)}')

fig, ax = plt.subplots(figsize=P.SQUARE_PANEL_SIZE_IN, dpi=P.DPI)
ax.scatter(gene_df['crop_seq'], gene_df['cell_dino'],
           s=6, color=POINT_COLOR, alpha=0.45, linewidths=0)
ax.plot([0, 1], [0, 1], linestyle='--', color=DIAG_COLOR, linewidth=0.8, alpha=0.7)

gene_df = gene_df.assign(diff=(gene_df['crop_seq'] - gene_df['cell_dino']).abs())
top = gene_df.nlargest(TOP_K, 'diff').reset_index(drop=True)
# texts = [
#     ax.text(row['crop_seq'], row['cell_dino'], row['perturbation'],
#             fontsize=LABEL_FONT_SIZE, color='black')
#     for _, row in top.iterrows()
# ]

ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
ax.set_xticks([0, 0.5, 1.0]); ax.set_yticks([0, 0.5, 1.0])
ax.set_xlabel('crop-seq mAP')
ax.set_ylabel('cell-dino mAP')
ax.set_title(f'Gene KO mAP (n={len(gene_df)})', fontsize=P.FONT_SIZE_TITLE)
ax.set_aspect('equal')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# adjust_text(
#     texts, ax=ax,
#     arrowprops=dict(arrowstyle='-', color='black', lw=0.4, shrinkA=2, shrinkB=2),
#     expand=(1.6, 1.6),
#     force_text=(0.6, 0.6),
#     force_static=(0.4, 0.4),
#     force_pull=(0.02, 0.02),
#     max_move=(40, 40),
# )

fig.tight_layout()
save_panel(fig, 'mAP_KO_scatter')
plt.show()

In [ ]:
# ---- GSEA on per-gene mAP difference (image vs crop-seq) ----
import gseapy as gp

# Ranking metric: the RAW mAP difference, cell_dino - crop_seq. Each gene's score says
# how much better one modality resolves that gene from the rest of the library, on the
# mAP scale, and the sign is meaningful in absolute terms. GSEA's negative tail is
# therefore exactly the set of genes where crop-seq genuinely wins — the sign check at
# the bottom of this cell asserts that no leading-edge gene of a negative-NES term has
# a positive difference.
#
# The systematic +0.13 offset (cell-dino resolves 704/1000 genes better) is real
# signal, not an artefact to normalise away.
#
#   'rank' — difference of within-modality percentile ranks — is retained below as a
#   diagnostic only. It removes the offset and is markedly more sensitive, but it
#   breaks the sign guarantee: a gene that imaging resolves better in absolute terms
#   can land on the RNA side if its crop-seq percentile happens to be higher.
METRIC = 'raw'                     # 'raw' drives the figure; 'rank' is diagnostic

scores = {
    'raw':  gene_df['cell_dino'] - gene_df['crop_seq'],
    'rank': gene_df['cell_dino'].rank(pct=True) - gene_df['crop_seq'].rank(pct=True),
}

# GO slim (goslim_generic), not the full GO DAG. Against this 1000-gene library the
# full Enrichr GO BP+CC gives 289 testable sets of which ~30% are near-duplicates of
# each other (Jaccard>=0.5) and the median set holds only 16 library genes — too few
# to detect the diffuse imaging-side signal. The slim gives 67 sets, 3% redundant,
# median 32 genes, covering 940/1000 genes.
# Built from GO release 2026-06-15 + goa_human.gaf (2026-05-21) via goatools.mapslim;
# see the header cell for the regeneration recipe.
LIBRARIES = {
    'GO_slim_BP': str(FIGURE_DATA / 'GO_slim_generic_BP_2026-06-15.gmt'),
    'GO_slim_CC': str(FIGURE_DATA / 'GO_slim_generic_CC_2026-06-15.gmt'),
}
pooled_sets, term_lib = {}, {}
for lib_name, gmt in LIBRARIES.items():
    with open(gmt) as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            pooled_sets[parts[0]] = [g for g in parts[2:] if g]
            term_lib[parts[0]] = lib_name

# BP and CC are pooled so the FDR is estimated once across all terms, rather than
# separately per panel (which under-counts the tests being run).
# max_size=200 drops the handful of slim terms that cover a fifth of the library.
MIN_SIZE, MAX_SIZE, N_PERM = 10, 200, 10_000


def run_prerank(score, tag):
    rnk = (gene_df.assign(score=score)[['perturbation', 'score']]
                  .sort_values('score', ascending=False)
                  .reset_index(drop=True))
    pre = gp.prerank(rnk=rnk, gene_sets=pooled_sets,
                     outdir=str(FIGURES_DIR / f'gsea_mAP_diff_{tag}'),
                     min_size=MIN_SIZE, max_size=MAX_SIZE, permutation_num=N_PERM,
                     seed=0, threads=8, verbose=False)
    res = pre.res2d.copy()
    for col in ('NES', 'NOM p-val', 'FDR q-val'):
        res[col] = pd.to_numeric(res[col], errors='coerce')
    res['lib'] = res['Term'].map(term_lib)
    return res.dropna(subset=['NES', 'FDR q-val']).reset_index(drop=True)


gsea_all = {tag: run_prerank(score, tag) for tag, score in scores.items()}

COLS = ['Term', 'lib', 'NES', 'NOM p-val', 'FDR q-val', 'Tag %']
for tag, res in gsea_all.items():
    n_bp = int((res['lib'] == 'GO_slim_BP').sum())
    n_cc = int((res['lib'] == 'GO_slim_CC').sum())
    flag = ' <-- FIGURE' if tag == METRIC else ' (diagnostic only)'
    print(f'\n===== {tag} metric — {len(res)} terms (BP={n_bp}, CC={n_cc}), '
          f'{N_PERM} permutations, pooled FDR ====={flag}')
    print(f'  min q = {res["FDR q-val"].min():.4f}    '
          + '   '.join(f'q<{t}: {int((res["FDR q-val"] < t).sum())}'
                       for t in (0.05, 0.10, 0.25)))
    for label, mask in (('RNA-favoured (NES<0)', res['NES'] < 0),
                        ('imaging-favoured (NES>0)', res['NES'] > 0)):
        sub = res[mask]
        print(f'  --- {label}: n={len(sub)}, '
              + ', '.join(f'q<{t}: {int((sub["FDR q-val"] < t).sum())}'
                          for t in (0.05, 0.25)) + ' ---')
        print(sub.nsmallest(6, 'FDR q-val')[COLS].to_string(index=False))

res_used = gsea_all[METRIC]

# Sign guarantee — the reason the raw difference is the published metric. Every gene
# driving a negative-NES term must be one where crop-seq genuinely beats cell-dino.
_diff = (gene_df.set_index('perturbation')['cell_dino']
         - gene_df.set_index('perturbation')['crop_seq'])
_neg = res_used[res_used['NES'] < 0]
_leak = sum(1 for _, row in _neg.iterrows()
            for gsym in str(row['Lead_genes']).split(';')
            if gsym in _diff.index and _diff[gsym] > 0)
print(f'\nsign check: {_leak} leading-edge genes across the {len(_neg)} negative-NES '
      f'terms have a positive mAP difference (must be 0)')
assert _leak == 0 or METRIC != 'raw'

In [ ]:
# ---- GSEA bar plot: top imaging vs RNA terms per library, coloured by FDR q ----
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.transforms import blended_transform_factory

Q_VMAX = 0.5                       # linear colour scale runs 0 -> 0.5; above that clips
LIB_TITLES = {'GO_slim_BP': 'GO Biological Process',
              'GO_slim_CC': 'GO Cellular Component'}

# This panel is read at a larger size than the mAP scatters, so it sets its own font
# sizes rather than inheriting the small paper-wide defaults from P.
FS_TERM, FS_TITLE, FS_AXIS, FS_TICK = 9, 12, 10, 9
FS_ARROW, FS_CBAR_LABEL, FS_CBAR_TICK = 9, 8, 7
Y_ARROW = -0.22                    # arrow row, in axes fraction below each panel

# Linear in q, not -log10(q): dark = small q. Truncated so the top of the range stays
# visible against white rather than fading to magma's near-white endpoint.
CMAP = LinearSegmentedColormap.from_list(
    'magma_trunc', plt.get_cmap('magma')(np.linspace(0.05, 0.85, 256)))
norm = plt.Normalize(vmin=0.0, vmax=Q_VMAX)

def _shorten(term, n=55):
    t = re.sub(r'\s*\(GO:\d+\)\s*$', '', term)
    return t if len(t) <= n else t[: n - 1] + '…'

def _select(res, k, strict_sign):
    """The k most imaging-leaning and k most RNA-leaning terms.

    strict_sign=True  — only NES>0 on the imaging side, NES<0 on the RNA side, so a
                        direction with few qualifying terms simply shows fewer bars.
    strict_sign=False — the k extremes at each end regardless of sign, so both sides
                        always show k terms even where nearly every NES is positive.
    """
    if strict_sign:
        pos = res[res['NES'] > 0].sort_values('NES', ascending=False).head(k)
        neg = res[res['NES'] < 0].sort_values('NES', ascending=True ).head(k)
    else:
        pos, neg = res.nlargest(k, 'NES'), res.nsmallest(k, 'NES')
    neg, pos = neg.assign(_rna_block=True), pos.assign(_rna_block=False)
    return (pd.concat([neg, pos], ignore_index=True)
              .drop_duplicates(subset='Term')
              .sort_values('NES').reset_index(drop=True))


def plot_gsea_panels(res_used, k, strict_sign, stem, axis_break=False, gap=1.3):
    panels = {lib: _select(res_used[res_used['lib'] == lib], k, strict_sign)
              for lib in LIBRARIES}
    n_max = max(len(p) for p in panels.values())

    extra = gap if axis_break else 0
    fig, axes = plt.subplots(1, 2, figsize=(10.0, 2.2 + 0.24 * (n_max + extra)), dpi=P.DPI,
                             gridspec_kw={'wspace': 0.30})
    fig.subplots_adjust(left=0.21, bottom=0.34, top=0.90)

    for i, (ax, lib_name) in enumerate(zip(axes, LIBRARIES)):
        plot_df = panels[lib_name]
        colors = CMAP(norm(plot_df['FDR q-val'].clip(upper=Q_VMAX)))
        y = np.arange(len(plot_df), dtype=float)
        # Rows are sorted by NES, so the RNA-side block is contiguous at the bottom.
        n_rna = int(plot_df['_rna_block'].sum())
        split = axis_break and 0 < n_rna < len(plot_df)
        if split:
            y[n_rna:] += gap

        ax.barh(y, plot_df['NES'], color=colors, linewidth=0)
        ax.axvline(0, color='#555555', linewidth=0.6)
        ax.set_yticks(y)
        ax.set_yticklabels([_shorten(t) for t in plot_df['Term']], fontsize=FS_TERM)
        ax.tick_params(axis='x', labelsize=FS_TICK)
        ax.tick_params(axis='y', length=2)
        ax.set_xlabel('NES', fontsize=FS_AXIS, labelpad=3)
        ax.set_title(LIB_TITLES[lib_name], fontsize=FS_TITLE, pad=8)
        ax.margins(x=0.06)
        ax.spines['top'].set_visible(False)
        if i == 1:                          # right plot: move tick labels to the right
            ax.yaxis.tick_right()
            ax.yaxis.set_label_position('right')
            ax.spines['left'].set_visible(False)
            ax.spines['right'].set_linewidth(0.6)
        else:
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_linewidth(0.6)

        if split:
            ax.set_ylim(y.min() - 0.7, y.max() + 0.7)
            # Broken-axis marks on the value spine, between the two blocks.
            y_br = (y[n_rna - 1] + y[n_rna]) / 2
            x_sp = 0.0 if i == 0 else 1.0
            trans = blended_transform_factory(ax.transAxes, ax.transData)
            ax.plot([x_sp, x_sp], [y_br - 0.30, y_br + 0.30], transform=trans,
                    color='white', lw=2.2, clip_on=False, zorder=4)
            for off in (-0.16, 0.16):
                ax.plot([x_sp - 0.013, x_sp + 0.013],
                        [y_br + off - 0.20, y_br + off + 0.20], transform=trans,
                        color='black', lw=0.9, clip_on=False,
                        solid_capstyle='butt', zorder=5)

        # Direction key beneath the axis: outward arrows flanked by the two modalities.
        for x_end in (0.28, 0.72):
            ax.annotate('', xy=(x_end, Y_ARROW), xytext=(0.50, Y_ARROW),
                        xycoords='axes fraction',
                        arrowprops=dict(arrowstyle='->', lw=1.0, color='black',
                                        shrinkA=0, shrinkB=0))
        ax.text(0.25, Y_ARROW, 'higher\nscRNA mAP', transform=ax.transAxes,
                ha='right', va='center', fontsize=FS_ARROW)
        ax.text(0.75, Y_ARROW, 'higher\nimaging mAP', transform=ax.transAxes,
                ha='left', va='center', fontsize=FS_ARROW)

    # Colour bar: vertically centred on the arrow row, horizontally centred on the
    # left panel's term labels. Both anchors are measured, not hardcoded.
    sm = plt.cm.ScalarMappable(norm=norm, cmap=CMAP)
    sm.set_array([])
    CB_W, CB_H = 0.095, 0.024
    fig.canvas.draw()                           # required before text extents are real
    _rend = fig.canvas.get_renderer()
    _inv = fig.transFigure.inverted()
    _lab = [t.get_window_extent(_rend) for t in axes[0].get_yticklabels()]
    _lx0 = _inv.transform((min(b.x0 for b in _lab), 0))[0]
    _lx1 = _inv.transform((max(b.x1 for b in _lab), 0))[0]
    _pos = axes[0].get_position()
    _cb_y = _pos.y0 + Y_ARROW * _pos.height     # figure y of the arrow row
    cax = fig.add_axes([(_lx0 + _lx1) / 2 - CB_W / 2, _cb_y - CB_H / 2, CB_W, CB_H])
    cbar = fig.colorbar(sm, cax=cax, orientation='horizontal', extend='max')
    cbar.set_ticks([0.0, 0.25, 0.5])
    cbar.set_label('FDR q-val', fontsize=FS_CBAR_LABEL, labelpad=2)
    cbar.ax.tick_params(labelsize=FS_CBAR_TICK, length=2, pad=1)
    cbar.outline.set_linewidth(0.5)

    save_panel(fig, stem)
    plt.show()
    return panels


# Main panel: up to 8 per direction, sign-restricted. The RNA side is short by
# construction because few terms have a negative NES at all.
plot_gsea_panels(res_used, k=8, strict_sign=True, stem=f'gsea_top_terms_{METRIC}')

# Balanced variant: the 4 extremes at each end regardless of sign or significance, so
# both modalities always get 4 rows. Rows on the RNA side may have a POSITIVE NES —
# those are the least imaging-leaning terms, not terms where crop-seq wins.
_panels = plot_gsea_panels(res_used, k=4, strict_sign=False, stem=f'gsea_top4_{METRIC}',
                           axis_break=True)
print('\ntop-4-each-end variant:')
for lib, df in _panels.items():
    neg = df[df['NES'] < 0]
    print(f'  {lib}: {len(df)} rows, {len(neg)} with NES<0 '
          f'(RNA side spans NES {df["NES"].min():+.2f} to {df["NES"].iloc[3]:+.2f})')